In [1]:
import pandas as pd
from math import ceil
import os
from sklearn.preprocessing import StandardScaler
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.multioutput import MultiOutputClassifier
import random
from sklearn.utils import resample

SEED =42 


random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)


def label_decode(np_data):
    person = []
    label = []
    for i in np_data:
        if i ==0 : 
            person.append(0)
            label.append(0)
        if i ==1 :
            person.append(0)
            label.append(0)
        if i ==2 :
            person.append(1)
            label.append(0)
        if i ==3 :
            person.append(1)
            label.append(1)
    return np.array(person) , np.array(label)

In [2]:
x_train = np.load("cnnout/train_features.npy")
y_train = np.load("cnnout/train_labels.npy")
y_train_person , y_train_label = label_decode(y_train)

x_test = np.load("cnnout/test_features.npy")
y_test = np.load("cnnout/test_labels.npy")
y_test_person , y_test_label = label_decode(y_test)

x_different = np.load("cnnout/different_features.npy")
y_different = np.load("cnnout/different_labels.npy")
y_different_person , y_different_label = label_decode(y_different)

print(np.unique(y_train))

[0 1 2 3]


In [ ]:
person_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.05,
    depth=4,          # 6'dan 4'e
    l2_leaf_reg=10,   # regularization ekle
    verbose=100,
    random_seed=SEED
)

# Target için class_weights ile azınlık sınıfına ağırlık
# Örnek: non-target=1, target=5
target_model = CatBoostClassifier(
    iterations=5000,
    learning_rate=0.05,
    depth=4,          # 6'dan 4'e
    l2_leaf_reg=10,   # regularization ekle
    verbose=100,
    class_weights=[1,3],
    random_seed=SEED
)


person_model.fit(x_train, y_train_person)
target_model.fit(x_train, y_train_label)
person_pred = person_model.predict(x_test)
target_pred = target_model.predict(x_test)

# Accuracy ayrı ayrı hesapla

print("Person accuracy:", accuracy_score(y_test_person, person_pred))
print("Target accuracy:", accuracy_score(y_test_label, target_pred))
print(classification_report(y_test_label, target_pred))
print(classification_report(y_test_person, person_pred))

0:	learn: 0.5482132	total: 143ms	remaining: 42.7s
100:	learn: 0.0005792	total: 1.02s	remaining: 2.02s
200:	learn: 0.0005557	total: 1.9s	remaining: 938ms
299:	learn: 0.0005550	total: 2.81s	remaining: 0us
0:	learn: 0.6051006	total: 9.23ms	remaining: 9.22s
100:	learn: 0.0808538	total: 938ms	remaining: 8.35s
200:	learn: 0.0622421	total: 1.9s	remaining: 7.56s
300:	learn: 0.0494328	total: 2.81s	remaining: 6.51s
400:	learn: 0.0402951	total: 3.71s	remaining: 5.55s
500:	learn: 0.0333114	total: 4.63s	remaining: 4.61s
600:	learn: 0.0279148	total: 5.55s	remaining: 3.68s
700:	learn: 0.0238475	total: 6.46s	remaining: 2.76s
800:	learn: 0.0209208	total: 7.39s	remaining: 1.83s
900:	learn: 0.0184227	total: 8.32s	remaining: 915ms
999:	learn: 0.0165473	total: 9.25s	remaining: 0us
Person accuracy: 1.0
Target accuracy: 0.9407692307692308
              precision    recall  f1-score   support

           0       0.98      0.94      0.96       980
           1       0.84      0.94      0.89       320

    accu

In [4]:
person_pred2 = person_model.predict(x_different)
target_pred2 = target_model.predict(x_different)

# Accuracy ayrı ayrı hesapla
#y_different_person , y_different_label
print("Person accuracy:", accuracy_score(y_different_person, person_pred2))
print("Target accuracy:", accuracy_score(y_different_label, target_pred2))
print(classification_report(y_different_label, target_pred2))
print(classification_report(y_different_person, person_pred2))

Person accuracy: 0.993993993993994
Target accuracy: 0.9301801801801802
              precision    recall  f1-score   support

           0       0.99      0.94      0.96      2450
           1       0.54      0.87      0.67       214

    accuracy                           0.93      2664
   macro avg       0.76      0.90      0.81      2664
weighted avg       0.95      0.93      0.94      2664

              precision    recall  f1-score   support

           0       1.00      0.99      0.99      1380
           1       0.99      1.00      0.99      1284

    accuracy                           0.99      2664
   macro avg       0.99      0.99      0.99      2664
weighted avg       0.99      0.99      0.99      2664



In [6]:
person_pred3 = person_model.predict(x_train)
target_pred3 = target_model.predict(x_train)

# Accuracy ayrı ayrı hesapla
#y_different_person , y_different_label
print("Person accuracy:", accuracy_score(y_train_person, person_pred3))
print("Target accuracy:", accuracy_score(y_train_label, target_pred3))
print(classification_report(y_train_label, target_pred3))
print(classification_report(y_train_person, person_pred3))

Person accuracy: 1.0
Target accuracy: 0.9980769230769231
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      3918
           1       0.99      1.00      1.00      1282

    accuracy                           1.00      5200
   macro avg       1.00      1.00      1.00      5200
weighted avg       1.00      1.00      1.00      5200

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2636
           1       1.00      1.00      1.00      2564

    accuracy                           1.00      5200
   macro avg       1.00      1.00      1.00      5200
weighted avg       1.00      1.00      1.00      5200

